# Live Ball Tracking with YOLO26m and ByteTrack

This notebook replaces the legacy OpenCV-only demo with a real notebook-first workflow for **sports-ball detection plus tracking**.

What this notebook does:
- downloads **COCO 2017** from Kaggle inside notebook code cells
- filters the dataset to the real **sports ball** category (COCO class 32)
- converts the subset into YOLO detection format
- fine-tunes **YOLO26m** with a **YOLO26s** fallback for constrained hardware
- evaluates the detector with real detection metrics
- runs **ByteTrack** on a real downloadable sports clip
- saves `best.pt`, `best.onnx`, `metrics.json`, `artifact_manifest.json`, and visual outputs

## 1. Problem Type and Why This Method Is Correct

**Task family:** object detection plus multi-object tracking.

Why this is the correct method:
- the project goal is to localize a ball and keep a stable identity across frames
- YOLO26m is the April 2026 default for trainable detection tasks
- ByteTrack is a practical tracker for associating detections frame to frame
- COCO 2017 contains the real `sports ball` category, so it is a grounded dataset choice for detector training

This notebook is honest about scope: COCO gives us **real detection supervision**, while the tracking section is a **video inference demonstration** using the trained detector plus ByteTrack.

## 2. April 2026 Best-Fit Tech Stack

- Core framework: Ultralytics YOLO
- Detector: YOLO26m, with YOLO26s fallback on lower-VRAM hardware
- Tracker: ByteTrack via `model.track(..., tracker='bytetrack.yaml')`
- Dataset source: Kaggle `awsaf49/coco-2017-dataset`
- Demo video source: Pexels sports clip
- Visualization: matplotlib + OpenCV
- Evaluation: Ultralytics validation metrics plus explicit exported JSON
- Export: PyTorch weights and ONNX

## 3. Dataset Options and Final Choice

Real dataset options for this task family:
- Kaggle `awsaf49/coco-2017-dataset`
- Roboflow football or sports-ball detection datasets
- dedicated possession / player-ball datasets for downstream sports analytics

**Final dataset choice:** COCO 2017 from Kaggle.

Why:
- it is public, stable, and downloadable in notebook cells
- it contains the real `sports ball` category needed for detector training
- it is a safer grounded choice than repo-local missing assets or synthetic fallback data

Dataset source: https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset

Demo video source: Pexels free stock video, used only for qualitative tracking inference.
Pexels clip page: https://www.pexels.com/video/man-training-soccer-on-a-soccer-field-7187049/

## 4. Environment Setup and Hardware Check

The next cell sets seeds, detects hardware, and keeps the notebook reproducible where practical.

In [1]:
import os
import platform
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Python  : {platform.python_version()}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM GB : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}')

Python  : 3.13.12
PyTorch : 2.6.0+cu124
CUDA    : True
Device  : cuda
GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM GB : 8.59


## 5. Dependency Installation

These installs are explicit and idempotent. No hidden setup is assumed.

In [3]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure_package('ultralytics')
ensure_package('cv2', 'opencv-python')
ensure_package('matplotlib')
ensure_package('pandas')
ensure_package('pycocotools')
ensure_package('kagglehub')
print('Dependencies satisfied.')

CalledProcessError: Command '['e:\\Github\\Machine-Learning-Projects\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'ultralytics']' returned non-zero exit status 1.

## 6. Imports and Project Paths

Everything stays local to this project folder so the notebook is self-contained.

In [1]:
import json
import shutil
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
WORK_DIR = PROJECT_DIR / 'working'
DATA_ROOT = WORK_DIR / 'data'
COCO_WORK_DIR = DATA_ROOT / 'coco_sports_ball'
VIDEO_DIR = DATA_ROOT / 'video'
RUNS_DIR = PROJECT_DIR / 'runs'

for folder in [ARTIFACTS_DIR, WORK_DIR, DATA_ROOT, COCO_WORK_DIR, VIDEO_DIR, RUNS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_DIR  : {PROJECT_DIR}')
print(f'ARTIFACTS_DIR: {ARTIFACTS_DIR}')
print(f'DATA_ROOT    : {DATA_ROOT}')

PROJECT_DIR  : e:\Github\Machine-Learning-Projects\Computer Vision\Project 18 - Live Ball Tracking
ARTIFACTS_DIR: e:\Github\Machine-Learning-Projects\Computer Vision\Project 18 - Live Ball Tracking\artifacts
DATA_ROOT    : e:\Github\Machine-Learning-Projects\Computer Vision\Project 18 - Live Ball Tracking\working\data


## 7. Real Download Rules

This notebook downloads two real assets:
- COCO 2017 from Kaggle for training and evaluation
- a real Pexels soccer clip for qualitative tracking inference

Credential rules for Kaggle:
- set `KAGGLE_USERNAME` and `KAGGLE_KEY`, or
- place `kaggle.json` in `~/.kaggle/`

If Kaggle credentials are missing, the notebook raises a clear error instead of fabricating fallback data.

In [ ]:
KAGGLE_DATASET = 'awsaf49/coco-2017-dataset'
PEXELS_VIDEO_URL = 'https://videos.pexels.com/video-files/7187049/7187049-hd_1920_1080_24fps.mp4'
VIDEO_PATH = VIDEO_DIR / 'ball_tracking_demo.mp4'

def check_kaggle_credentials():
    has_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    has_file = (Path.home() / '.kaggle' / 'kaggle.json').exists()
    if not has_env and not has_file:
        raise RuntimeError('Kaggle credentials not found. Set KAGGLE_USERNAME/KAGGLE_KEY or create ~/.kaggle/kaggle.json')

def download_coco_dataset():
    check_kaggle_credentials()
    try:
        import kagglehub
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception:
        subprocess.check_call(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(COCO_WORK_DIR), '--unzip'])
        return COCO_WORK_DIR

def download_demo_video():
    if VIDEO_PATH.exists() and VIDEO_PATH.stat().st_size > 0:
        return VIDEO_PATH
    urllib.request.urlretrieve(PEXELS_VIDEO_URL, VIDEO_PATH)
    return VIDEO_PATH

raw_root = Path(download_coco_dataset())
video_path = download_demo_video()
print(f'Raw dataset root: {raw_root}')
print(f'Demo video path : {video_path}')
print(f'Demo video MB   : {video_path.stat().st_size / (1024 * 1024):.2f}')

## 8. Dataset Verification and Integrity Checks

We first locate the actual COCO directory layout, then verify image folders, annotation files, and the demo video.

In [ ]:
def find_coco_root(root):
    candidates = [root]
    if root.exists():
        candidates.extend([child for child in root.iterdir() if child.is_dir()])
    for candidate in candidates:
        train_dir = candidate / 'train2017'
        val_dir = candidate / 'val2017'
        ann_dir = candidate / 'annotations'
        if train_dir.exists() and val_dir.exists() and ann_dir.exists():
            return candidate
        img_train = candidate / 'images' / 'train2017'
        img_val = candidate / 'images' / 'val2017'
        if img_train.exists() and img_val.exists() and ann_dir.exists():
            return candidate
    raise RuntimeError(f'Could not locate COCO directory structure under {root}')

COCO_ROOT = find_coco_root(raw_root)
TRAIN_IMG_ROOT = COCO_ROOT / 'train2017'
VAL_IMG_ROOT = COCO_ROOT / 'val2017'
if not TRAIN_IMG_ROOT.exists():
    TRAIN_IMG_ROOT = COCO_ROOT / 'images' / 'train2017'
if not VAL_IMG_ROOT.exists():
    VAL_IMG_ROOT = COCO_ROOT / 'images' / 'val2017'
ANN_DIR = COCO_ROOT / 'annotations'
TRAIN_JSON = ANN_DIR / 'instances_train2017.json'
VAL_JSON = ANN_DIR / 'instances_val2017.json'

assert TRAIN_IMG_ROOT.exists(), f'Missing train images: {TRAIN_IMG_ROOT}'
assert VAL_IMG_ROOT.exists(), f'Missing val images: {VAL_IMG_ROOT}'
assert TRAIN_JSON.exists(), f'Missing train json: {TRAIN_JSON}'
assert VAL_JSON.exists(), f'Missing val json: {VAL_JSON}'
assert video_path.exists() and video_path.stat().st_size > 0, 'Demo video download failed'

print(f'COCO root        : {COCO_ROOT}')
print(f'Train images dir : {TRAIN_IMG_ROOT}')
print(f'Val images dir   : {VAL_IMG_ROOT}')
print(f'Train json       : {TRAIN_JSON.name}')
print(f'Val json         : {VAL_JSON.name}')

## 9. Label Inspection and Sports-Ball Sampling Strategy

COCO class `32` is `sports ball`. We keep only images that contain at least one ball annotation.

Practical subset targets:
- up to 1800 training images
- 300 validation images
- 150 test images

This keeps the project runnable on local hardware while remaining grounded in real annotations.

In [ ]:
with open(TRAIN_JSON, 'r', encoding='utf-8') as f:
    train_meta = json.load(f)
with open(VAL_JSON, 'r', encoding='utf-8') as f:
    val_meta = json.load(f)

COCO_SPORTS_BALL_ID = 32
class_name = {item['id']: item['name'] for item in train_meta['categories']}[COCO_SPORTS_BALL_ID]
assert class_name == 'sports ball', f'Expected sports ball, found {class_name}'

def build_subset_index(meta):
    images_by_id = {row['id']: row for row in meta['images']}
    anns_by_image = defaultdict(list)
    for ann in meta['annotations']:
        if ann.get('iscrowd', 0) == 1:
            continue
        if ann['category_id'] != COCO_SPORTS_BALL_ID:
            continue
        x, y, w, h = ann['bbox']
        if w <= 1 or h <= 1:
            continue
        anns_by_image[ann['image_id']].append(ann)
    selected_ids = sorted(anns_by_image.keys())
    return images_by_id, anns_by_image, selected_ids

train_images_by_id, train_anns_by_image, train_selected_ids = build_subset_index(train_meta)
val_images_by_id, val_anns_by_image, val_selected_ids = build_subset_index(val_meta)

print(f'Sports-ball train images available: {len(train_selected_ids):,}')
print(f'Sports-ball val images available  : {len(val_selected_ids):,}')
print(f'Sports-ball train annotations     : {sum(len(v) for v in train_anns_by_image.values()):,}')
print(f'Sports-ball val annotations       : {sum(len(v) for v in val_anns_by_image.values()):,}')

## 10. Exploratory Data Analysis

We inspect sample images and box-size statistics to understand the real difficulty of the task. Sports balls are often small, blurred, and partially occluded.

In [ ]:
sample_train_ids = train_selected_ids[:9]
box_areas = []
for image_id in train_selected_ids[:500]:
    img_meta = train_images_by_id[image_id]
    img_area = img_meta['width'] * img_meta['height']
    for ann in train_anns_by_image[image_id]:
        box_area = ann['bbox'][2] * ann['bbox'][3]
        box_areas.append(box_area / img_area)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, image_id in zip(axes.flatten(), sample_train_ids):
    img_meta = train_images_by_id[image_id]
    img = cv2.imread(str(TRAIN_IMG_ROOT / img_meta['file_name']))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for ann in train_anns_by_image[image_id]:
        x, y, w, h = ann['bbox']
        x1, y1, x2, y2 = int(x), int(y), int(x + w), int(y + h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 80, 80), 2)
    ax.imshow(img)
    ax.set_title(img_meta['file_name'])
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'sample_training_images.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(box_areas, bins=30, color='#1f77b4', edgecolor='white')
plt.title('Relative sports-ball box area distribution')
plt.xlabel('bbox area / image area')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'bbox_area_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Train / Validation / Test Split Strategy

COCO already separates train and validation data, so we avoid leakage by keeping training images from `train2017` only.

We then carve the validation set into two honest slices:
- one for validation during training
- one held out for final detector testing

In [ ]:
MAX_TRAIN_IMAGES = min(1800, len(train_selected_ids))
MAX_VAL_IMAGES = min(300, max(0, len(val_selected_ids) - 150))
MAX_TEST_IMAGES = min(150, len(val_selected_ids) - MAX_VAL_IMAGES)
if MAX_VAL_IMAGES <= 0 or MAX_TEST_IMAGES <= 0:
    raise RuntimeError('Not enough sports-ball images in the COCO validation set to build val/test splits.')

train_subset_ids = train_selected_ids[:MAX_TRAIN_IMAGES]
val_subset_ids = val_selected_ids[:MAX_VAL_IMAGES]
test_subset_ids = val_selected_ids[MAX_VAL_IMAGES:MAX_VAL_IMAGES + MAX_TEST_IMAGES]

print(f'Train subset size: {len(train_subset_ids)}')
print(f'Val subset size  : {len(val_subset_ids)}')
print(f'Test subset size : {len(test_subset_ids)}')

## 12. Preprocessing Strategy and YOLO Conversion

The target is a **single-class detector**:
- class `0` in YOLO format becomes `sports_ball`
- only real sports-ball annotations are kept
- images without a sports-ball instance are excluded from this training notebook

This is a deliberate design choice because the project goal is ball tracking, not generic COCO detection.

In [ ]:
YOLO_DATASET_DIR = COCO_WORK_DIR / 'yolo_sports_ball'

def prepare_split(split_name, image_root, images_by_id, anns_by_image, selected_ids):
    img_out = YOLO_DATASET_DIR / split_name / 'images'
    lbl_out = YOLO_DATASET_DIR / split_name / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for image_id in selected_ids:
        img_meta = images_by_id[image_id]
        src_img = image_root / img_meta['file_name']
        dst_img = img_out / img_meta['file_name']
        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)

        width = img_meta['width']
        height = img_meta['height']
        label_lines = []
        for ann in anns_by_image[image_id]:
            x, y, w, h = ann['bbox']
            x_center = (x + w / 2.0) / width
            y_center = (y + h / 2.0) / height
            w_norm = w / width
            h_norm = h / height
            label_lines.append(f'0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}')

        (lbl_out / f"{Path(img_meta['file_name']).stem}.txt").write_text('\n'.join(label_lines), encoding='utf-8')

prepare_split('train', TRAIN_IMG_ROOT, train_images_by_id, train_anns_by_image, train_subset_ids)
prepare_split('val', VAL_IMG_ROOT, val_images_by_id, val_anns_by_image, val_subset_ids)
prepare_split('test', VAL_IMG_ROOT, val_images_by_id, val_anns_by_image, test_subset_ids)

data_yaml = YOLO_DATASET_DIR / 'data.yaml'
data_yaml.write_text(
    '\n'.join([
        f'path: {YOLO_DATASET_DIR.as_posix()}',
        'train: train/images',
        'val: val/images',
        'test: test/images',
        'nc: 1',
        "names: ['sports_ball']",
    ]),
    encoding='utf-8'
)

print(data_yaml.read_text(encoding='utf-8'))

## 13. Baseline Approach

Before fine-tuning, we run a quick baseline using a pretrained YOLO26s detector.

This is not the final model. It is a grounded baseline to check whether pretrained COCO knowledge is already enough for some ball detections.

In [ ]:
baseline_model = YOLO('yolo26s.pt')
baseline_preview = []
for image_id in test_subset_ids[:6]:
    img_meta = val_images_by_id[image_id]
    img_path = VAL_IMG_ROOT / img_meta['file_name']
    result = baseline_model.predict(source=str(img_path), conf=0.15, classes=[32], verbose=False)[0]
    rendered = result.plot()
    baseline_preview.append(cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, image in zip(axes.flatten(), baseline_preview):
    ax.imshow(image)
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'baseline_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Main Model Training Rules

Training decisions:
- start with YOLO26m for better small-object accuracy
- fall back to YOLO26s if the run fails on memory
- use a higher image size than generic object detection because sports balls are often tiny
- save everything into the project-local `runs/` directory

In [ ]:
TRAIN_EPOCHS = 20
TRAIN_IMGSZ = 960
TRAIN_BATCH = 8 if DEVICE == 'cuda' else 4
PRIMARY_MODEL = 'yolo26m.pt'
FALLBACK_MODEL = 'yolo26s.pt'

selected_model_name = PRIMARY_MODEL
trainer = YOLO(selected_model_name)
try:
    train_results = trainer.train(
        data=str(data_yaml),
        epochs=TRAIN_EPOCHS,
        imgsz=TRAIN_IMGSZ,
        batch=TRAIN_BATCH,
        project=str(RUNS_DIR),
        name='live_ball_tracking',
        exist_ok=True,
        device=0 if DEVICE == 'cuda' else 'cpu',
        seed=SEED,
        workers=2,
        pretrained=True,
        verbose=True,
    )
except RuntimeError as exc:
    if 'out of memory' not in str(exc).lower():
        raise
    selected_model_name = FALLBACK_MODEL
    trainer = YOLO(selected_model_name)
    train_results = trainer.train(
        data=str(data_yaml),
        epochs=TRAIN_EPOCHS,
        imgsz=TRAIN_IMGSZ,
        batch=max(2, TRAIN_BATCH // 2),
        project=str(RUNS_DIR),
        name='live_ball_tracking',
        exist_ok=True,
        device=0 if DEVICE == 'cuda' else 'cpu',
        seed=SEED,
        workers=2,
        pretrained=True,
        verbose=True,
    )

run_dir = Path(trainer.trainer.save_dir)
best_weights = run_dir / 'weights' / 'best.pt'
assert best_weights.exists(), f'Missing best weights: {best_weights}'
print(f'Model used : {selected_model_name}')
print(f'Run dir    : {run_dir}')
print(f'Best weight: {best_weights}')

## 15. Detector Evaluation Rules

For this project the correct detection metrics are:
- precision
- recall
- mAP50
- mAP50-95

Tracking quality is reviewed qualitatively in the video section because this notebook does not create a labeled multi-frame tracking benchmark.

In [ ]:
best_model = YOLO(str(best_weights))
val_results = best_model.val(data=str(data_yaml), split='test', imgsz=TRAIN_IMGSZ, batch=max(1, TRAIN_BATCH // 2), device=0 if DEVICE == 'cuda' else 'cpu', verbose=False)
metrics = {
    'model_used': selected_model_name,
    'train_epochs': TRAIN_EPOCHS,
    'imgsz': TRAIN_IMGSZ,
    'train_images': len(train_subset_ids),
    'val_images': len(val_subset_ids),
    'test_images': len(test_subset_ids),
    'precision': float(val_results.box.mp),
    'recall': float(val_results.box.mr),
    'map50': float(val_results.box.map50),
    'map50_95': float(val_results.box.map),
}

metrics_path = ARTIFACTS_DIR / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

## 16. Tracking Inference Rules

Now we switch from image-level detection to video-level tracking.

Rules for this section:
- use the trained detector if available
- keep only the sports-ball class
- run ByteTrack with persistent IDs
- save an annotated MP4 and a frame montage for inspection

In [ ]:
TRACKED_VIDEO_PATH = ARTIFACTS_DIR / 'tracked_demo.mp4'
FRAME_STRIDE = 20
MAX_TRACK_FRAMES = 240

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f'Could not open demo video: {video_path}')

fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(TRACKED_VIDEO_PATH), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
preview_frames = []
frame_idx = 0

while frame_idx < MAX_TRACK_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    results = best_model.track(frame, persist=True, tracker='bytetrack.yaml', conf=0.15, verbose=False)[0]
    annotated = results.plot()
    ids = results.boxes.id if results.boxes is not None else None
    tracked_count = int(len(ids)) if ids is not None else 0
    cv2.putText(annotated, f'Sports-ball tracks: {tracked_count}', (16, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    writer.write(annotated)
    if frame_idx % FRAME_STRIDE == 0 and len(preview_frames) < 6:
        preview_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    frame_idx += 1

cap.release()
writer.release()

print(f'Frames processed : {frame_idx}')
print(f'Tracked video out: {TRACKED_VIDEO_PATH}')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, image in zip(axes.flatten(), preview_frames):
    ax.imshow(image)
    ax.axis('off')
for ax in axes.flatten()[len(preview_frames):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'tracking_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 17. Error Analysis, Limitations, and Common Mistakes

Expected failure modes:
- very small distant balls
- heavy motion blur
- balls partially hidden by players
- sports domains not well represented by the COCO subset used here

Common mistakes when extending this project:
- training a generic detector but forgetting to filter to the ball class during tracking
- assuming image detection metrics are the same as tracking metrics
- using too small an image size for tiny balls
- evaluating tracking on a clip that was never actually downloaded or verified

How to improve this project:
- train on a sports-specific ball dataset or a player+ball dataset
- add a real MOT-style benchmark with track annotations
- compare ByteTrack vs BoT-SORT
- add trajectory smoothing and field-aware constraints

Production considerations:
- YOLO26s may be the better deployment choice for real-time latency
- camera angle and zoom matter heavily for small-ball recall
- downstream analytics such as possession need player detections too, not just the ball

Mini challenge:
- extend the notebook to detect both `person` and `sports ball`, then estimate nearest-player possession on the same clip

## 18. Artifact Saving Rules and Final Deliverables

The final cell exports the trained weights, ONNX model, metrics JSON, and a manifest so the project outputs are reviewable outside the notebook.

In [ ]:
exported_onnx = best_model.export(format='onnx')
exported_onnx = Path(exported_onnx)
final_best_pt = ARTIFACTS_DIR / 'best.pt'
final_best_onnx = ARTIFACTS_DIR / 'best.onnx'
shutil.copy2(best_weights, final_best_pt)
shutil.copy2(exported_onnx, final_best_onnx)

artifact_manifest = {
    'project': 'Live Ball Tracking',
    'dataset': KAGGLE_DATASET,
    'demo_video_url': PEXELS_VIDEO_URL,
    'selected_model': selected_model_name,
    'artifacts': {
        'best_pt': str(final_best_pt),
        'best_onnx': str(final_best_onnx),
        'metrics_json': str(metrics_path),
        'baseline_preview': str(ARTIFACTS_DIR / 'baseline_preview.png'),
        'sample_training_images': str(ARTIFACTS_DIR / 'sample_training_images.png'),
        'bbox_area_histogram': str(ARTIFACTS_DIR / 'bbox_area_histogram.png'),
        'tracking_preview': str(ARTIFACTS_DIR / 'tracking_preview.png'),
        'tracked_video': str(TRACKED_VIDEO_PATH),
    },
}

manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
manifest_path.write_text(json.dumps(artifact_manifest, indent=2), encoding='utf-8')
print(json.dumps(artifact_manifest, indent=2))

## 19. Final Summary

This notebook turns Project 18 into a modern notebook-first pipeline by:
- training a real single-class sports-ball detector from COCO 2017
- validating it with real detection metrics
- running ByteTrack on a real sports clip
- exporting reviewable artifacts for follow-up work

It is intentionally honest about what is and is not evaluated: detection is measured quantitatively, while tracking is demonstrated qualitatively on video because no multi-frame track labels are introduced here.